bai1:

In [ ]:
#Hàm 1
def activity_selection(activities):
    """
    Chọn số lượng hoạt động tối đa không chồng lấp
    Chiến lược: Chọn kết thúc sớm nhất (earliest finish time)
    """
    if not activities:
        return []

    # Bước 1: Sort theo finish time (index 1)
    activities.sort(key=lambda x: x[1])

    # Bước 2: Chọn hoạt động đầu tiên
    selected = [activities[0]]
    last_finish = activities[0][1]

    # Bước 3: Duyệt các hoạt động còn lại
    for i in range(1, len(activities)):
        start, finish = activities[i]
        
        # Nếu start time >= thời gian kết thúc của hoạt động trước đó → Chọn
        if start >= last_finish:
            selected.append((start, finish))
            last_finish = finish
            
    return selected

# Test case 1
print("=== Test Activity Selection ===")
activities1 = [
    (1, 4), (3, 5), (0, 6), (5, 7), (3, 9), 
    (5, 9), (6, 10), (8, 11), (8, 12), (2, 14), (12, 16)
]
result1 = activity_selection(activities1)
print(f"Hoạt động được chọn: {result1}")
print(f"Số lượng: {len(result1)}") 

# Test case 2
activities2 = [(1, 3), (2, 4), (3, 5), (4, 6)]
result2 = activity_selection(activities2)
print(f"\nHoạt động được chọn: {result2}")
print(f"Số lượng: {len(result2)}")

#hàm 2
def coin_change_greedy(amount, coins):
    """
    Đổi tiền bằng số xu ít nhất (Greedy approach)
    """
    # Bước 1: Sort giảm dần
    coins.sort(reverse=True)
    count = 0
    result = []
    
    # Bước 2: Duyệt từng mệnh giá
    for coin in coins:
        if amount == 0:
            break
        # Dùng phép chia lấy nguyên để lấy số lượng xu nhanh hơn thay vì while loop
        num_coins = amount // coin
        if num_coins > 0:
            count += num_coins
            amount %= coin
            result.extend([coin] * num_coins)
            
    # Bước 3: Kiểm tra kết quả
    if amount == 0:
        return count, result
    else:
        return -1, []

# Test
print("\n=== Test Coin Change Greedy ===")
print("Test 1: Hệ chuẩn [25, 10, 5, 1] - amount 63")
print(coin_change_greedy(63, [25, 10, 5, 1]))

print("\nTest 3: Hệ mệnh giá lạ [25, 10, 1] - amount 30")
count, res = coin_change_greedy(30, [25, 10, 1])
print(f"Greedy chọn: {res} (Tổng: {count} xu)")
print("⚠️ Kết quả tối ưu phải là: [10, 10, 10] (3 xu)")

#hàm 3
def fractional_knapsack(capacity, items):
    """
    Bài toán Ba lô phân số (Fractional Knapsack)
    """
    # Bước 1: Tính ratio (value/weight) cho mỗi vật
    items_with_ratio = []
    for weight, value in items:
        ratio = value / weight
        items_with_ratio.append((weight, value, ratio))
    
    # Sort theo ratio giảm dần (index 2 là ratio)
    items_with_ratio.sort(key=lambda x: x[2], reverse=True)
    
    # Bước 2: Duyệt và chọn
    total_value = 0.0
    remaining_capacity = capacity
    result = []
    
    for weight, value, ratio in items_with_ratio:
        if remaining_capacity <= 0:
            break
            
        if weight <= remaining_capacity:
            # Lấy hết vật phẩm
            total_value += value
            remaining_capacity -= weight
            result.append((weight, value, 1.0))
        else:
            # Lấy một phần của vật phẩm
            fraction = remaining_capacity / weight
            total_value += value * fraction
            result.append((weight, value, fraction))
            remaining_capacity = 0
            
    return total_value, result

# Test
print("\n=== Test Fractional Knapsack ===")
capacity1 = 50
items1 = [(10, 60), (20, 100), (30, 120)]
total1, result1 = fractional_knapsack(capacity1, items1)
print(f"Giá trị tối đa: {total1}")
for w, v, f in result1:
    print(f" Vật (w={w}, v={v}): Lấy {f*100:.1f}%")

#hàm 4
# Giả sử bạn đã có hàm activity_selection từ bài trước
def min_intervals_remove(intervals):
    """
    Tìm số khoảng thời gian ít nhất cần xóa 
    để các khoảng còn lại không chồng lấp.
    """
    if not intervals:
        return 0
        
    # Tính số lượng khoảng giữ lại tối đa bằng thuật toán Greedy
    max_keep = activity_selection(intervals)
    
    # Số cần xóa chính là phần bù
    num_remove = len(intervals) - len(max_keep)
    return num_remove

# Test
intervals1 = [(1, 2), (2, 3), (3, 4), (1, 3)]
print(f"Số khoảng cần xóa (Test 1): {min_intervals_remove(intervals1)}")

intervals2 = [(1, 2), (1, 2), (1, 2)]
print(f"Số khoảng cần xóa (Test 2): {min_intervals_remove(intervals2)}")

intervals3 = [(1, 100), (11, 22), (1, 11), (2, 12)]
print(f"Số khoảng cần xóa (Test 3): {min_intervals_remove(intervals3)}")




bai2:

In [1]:
import heapq
import time
import heapq

from lab4_bai1 import coin_change_greedy

def min_meeting_rooms(meetings):
    """
    Tìm số phòng họp tối thiểu cần thiết
    Chiến lược: Sort theo start time + dùng Heap theo dõi end time
    """
    if not meetings:
        return 0
    
    # 1. Sort meetings theo start time
    meetings.sort(key=lambda x: x[0])
    
    # Heap lưu end time của các phòng đang dùng
    heap = []
    
    # 2. Thêm end time của meeting đầu tiên vào heap
    heapq.heappush(heap, meetings[0][1])
    
    # 3. Duyệt các meetings còn lại
    for i in range(1, len(meetings)):
        start, end = meetings[i]
        
        # Nếu phòng sớm nhất đã trống (start >= end time cũ)
        # → Tái sử dụng phòng đó (cập nhật bằng cách pop và push)
        if start >= heap[0]:
            heapq.heappop(heap)
            
        # Thêm end time của meeting hiện tại vào heap
        heapq.heappush(heap, end)
        
    # Số phòng = số phần tử trong heap
    return len(heap)

# Test
meetings = [(0, 30), (5, 10), (15, 20)]
print(f"Số phòng tối thiểu cần thiết: {min_meeting_rooms(meetings)}")
# Kỳ vọng: 2

#Phần B
def coin_change_dp(amount, coins):
    """
    Coin Change với Dynamic Programming
    Luôn cho lời giải tối ưu
    """
    # Khởi tạo dp array với giá trị vô cùng lớn
    dp = [float('inf')] * (amount + 1)
    
    # Base case: 0 xu để đổi số tiền 0
    dp[0] = 0 
    
    # Tính dp[i] cho i từ 1 đến amount
    for i in range(1, amount + 1):
        for coin in coins:
            if coin <= i:
                # Công thức truy hồi: chọn giá trị nhỏ nhất giữa việc 
                # giữ nguyên hiện tại hoặc thêm 1 đồng xu vào cách đổi cũ
                dp[i] = min(dp[i], dp[i - coin] + 1)
                
    # Trả về kết quả
    return dp[amount] if dp[amount] != float('inf') else -1

# Test với hệ mệnh giá lạ (Greedy đã thất bại ở đây)
coins = [25, 10, 1]
amount = 30
result = coin_change_dp(amount, coins)
print(f"Số xu ít nhất để đổi {amount} là: {result}") 
# Kết quả: 3 (tương ứng với 10+10+10)

#PHần C
import time

def compare_coin_change(amount, coins):
    print(f"\n{'='*60}")
    print(f"So sánh Coin Change: amount={amount}, coins={coins}")
    print(f"{'='*60}")
    
    # 1. Test Greedy
    print("\n[1] GREEDY:")
    start = time.time()
    greedy_result, greedy_detail = coin_change_greedy(amount, coins)
    greedy_time = time.time() - start
    print(f"Kết quả: {greedy_result} xu")
    print(f"Chi tiết: {greedy_detail}")
    print(f"Thời gian: {greedy_time:.6f}s")
    
    # 2. Test DP
    print("\n[2] DYNAMIC PROGRAMMING:")
    start = time.time()
    dp_result = coin_change_dp(amount, coins)
    dp_time = time.time() - start
    print(f"Kết quả: {dp_result} xu")
    print(f"Thời gian: {dp_time:.6f}s")
    
    # 3. So sánh
    print("\n[3] SO SÁNH:")
    if greedy_result == dp_result:
        print("✅ Greedy ĐÚNG - cho kết quả tối ưu!")
    else:
        print(f"❌ Greedy SAI - kém hơn {dp_result - greedy_result} xu!")
        
    if greedy_time > 0:
        print(f"Tốc độ: Greedy nhanh hơn {dp_time/greedy_time:.2f}x")
    else:
        print("Tốc độ: Cả hai đều cực nhanh.")

# Test với các trường hợp
compare_coin_change(67, [25, 10, 5, 1])
compare_coin_change(30, [25, 10, 1])


=== Test Activity Selection ===
Hoạt động được chọn: [(1, 4), (5, 7), (8, 11), (12, 16)]
Số lượng: 4

Hoạt động được chọn: [(1, 3), (3, 5)]
Số lượng: 2

=== Test Coin Change Greedy ===
Test 1: Hệ chuẩn [25, 10, 5, 1] - amount 63
(6, [25, 25, 10, 1, 1, 1])

Test 3: Hệ mệnh giá lạ [25, 10, 1] - amount 30
Greedy chọn: [25, 1, 1, 1, 1, 1] (Tổng: 6 xu)
⚠️ Kết quả tối ưu phải là: [10, 10, 10] (3 xu)

=== Test Fractional Knapsack ===
Giá trị tối đa: 240.0
 Vật (w=10, v=60): Lấy 100.0%
 Vật (w=20, v=100): Lấy 100.0%
 Vật (w=30, v=120): Lấy 66.7%
Số khoảng cần xóa (Test 1): 1
Số khoảng cần xóa (Test 2): 2
Số khoảng cần xóa (Test 3): 2
Số phòng tối thiểu cần thiết: 2
Số xu ít nhất để đổi 30 là: 3

So sánh Coin Change: amount=67, coins=[25, 10, 5, 1]

[1] GREEDY:
Kết quả: 6 xu
Chi tiết: [25, 25, 10, 5, 1, 1]
Thời gian: 0.000005s

[2] DYNAMIC PROGRAMMING:
Kết quả: 6 xu
Thời gian: 0.000027s

[3] SO SÁNH:
✅ Greedy ĐÚNG - cho kết quả tối ưu!
Tốc độ: Greedy nhanh hơn 5.00x

So sánh Coin Change: amount

bai3:


In [2]:
def find_content_children(greed, cookies):
    """
    Assign Cookies - LeetCode 455
    Chiến lược: Sort cả 2 mảng + Two Pointers
    """
    # TODO: Sort cả greed và cookies
    greed.sort()
    cookies.sort()
    
    # TODO: Dùng 2 con trỏ duyệt
    i, j = 0, 0
    content_children = 0
    
    while i < len(greed) and j < len(cookies):
        if cookies[j] >= greed[i]:
            content_children += 1
            i += 1
        j += 1
    
    # TODO: Trả về số trẻ đã thỏa mãn
    return content_children
pass

#Phần C
def manhattan_distance(p1, p2):
    return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])

def assign_bikes(workers, bikes):
    """
    Ghép worker với bike bằng chiến lược Greedy
    """
    # 1. Tính tất cả khoảng cách Manhattan
    distances = []
    for i, w in enumerate(workers):
        for j, b in enumerate(bikes):
            dist = manhattan_distance(w, b)
            distances.append((dist, i, j)) # (khoảng cách, worker_idx, bike_idx)
    
    # 2. Sort theo khoảng cách tăng dần
    distances.sort(key=lambda x: x[0])
    
    # 3. Greedy chọn cặp gần nhất chưa dùng
    worker_used = [False] * len(workers)
    bike_used = [False] * len(bikes)
    assignments = []
    total_distance = 0
    
    for dist, w_idx, b_idx in distances:
        if not worker_used[w_idx] and not bike_used[b_idx]:
            worker_used[w_idx] = True
            bike_used[b_idx] = True
            assignments.append((w_idx, b_idx, dist))
            total_distance += dist
            
    return total_distance, assignments

# Test
workers = [(0, 0), (2, 1)]
bikes = [(1, 2), (3, 3)]
total, results = assign_bikes(workers, bikes)

print(f"Tổng khoảng cách nhỏ nhất (Greedy): {total}")
for w, b, dist in results:
    print(f"Worker {w} ghép với Bike {b} (Khoảng cách: {dist})")

#Phần C
import random
import time
import matplotlib.pyplot as plt

def activity_selection(activities):
    """
    Chọn tập hoạt động tối đa không giao nhau bằng chiến lược Greedy.
    activities: list of (start, end)
    Trả về list các hoạt động đã chọn (start, end), sắp theo thời gian kết thúc.
    """
    if not activities:
        return []
    # Sort activities by end time
    activities_sorted = sorted(activities, key=lambda x: x[1])
    selected = [activities_sorted[0]]
    last_end = activities_sorted[0][1]
    for s, e in activities_sorted[1:]:
        if s >= last_end:
            selected.append((s, e))
            last_end = e
    return selected

# Giả sử hàm activity_selection đã được định nghĩa ở bước trước
def generate_test_data(n):
    """Tạo dữ liệu test ngẫu nhiên"""
    activities = []
    for i in range(n):
        start = random.randint(0, 1000)
        duration = random.randint(1, 50)
        end = start + duration
        activities.append((start, end))
    return activities

def benchmark_activity_selection(sizes):
    """Đo thời gian với các kích thước input khác nhau"""
    times = []
    print(f"{'Kích thước (n)':<15} | {'Thời gian (s)':<15}")
    print("-" * 35)
    
    for size in sizes:
        activities = generate_test_data(size)

        start = time.time()
        activity_selection(activities)
        elapsed = time.time() - start
        
        times.append(elapsed)
        print(f"{size:<15} | {elapsed:.6f}s")
    return times

# Chạy benchmark
sizes = [1000, 5000, 10000, 50000, 100000]
times = benchmark_activity_selection(sizes)

# Vẽ biểu đồ để thấy rõ độ phức tạp O(n log n)
plt.figure(figsize=(8, 5))
plt.plot(sizes, times, marker='o', linestyle='-', color='b')
plt.title("Hiệu năng thuật toán Activity Selection (O(n log n))")
plt.xlabel("Số lượng hoạt động (n)")
plt.ylabel("Thời gian chạy (giây)")
plt.grid(True)
plt.show()

Tổng khoảng cách nhỏ nhất (Greedy): 8
Worker 1 ghép với Bike 0 (Khoảng cách: 2)
Worker 0 ghép với Bike 1 (Khoảng cách: 6)


ModuleNotFoundError: No module named 'matplotlib'